In [1]:
import os
import sys
project_dir = os.path.dirname(os.getcwd())
sys.path.append(project_dir)

from utils.summary import get_model_stats

import torch
import torch.nn.functional as F

In [2]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cpu


In [3]:
import models.baseline as mlp

teacher = mlp.mnist1200().to(device)
student = mlp.mnist800().to(device)

sample = torch.randn(1, 1, 28, 28).to(device)
with torch.no_grad():
    print("Teacher model stats:")
    pred = teacher(sample)
    for name, param in get_model_stats(teacher, sample.shape).items():
        print(f"{name}: {param}")
    print("Student model stats:")
    pred = student(sample)
    for name, param in get_model_stats(student, sample.shape).items():
        print(f"{name}: {param}")

Teacher model stats:
flops: 3832800
params: 3836410
Student model stats:
flops: 1915200
params: 1917610


In [4]:
from models.segmentation import TeacherModel
from models.segmentation import StudentModel

teacher = TeacherModel(in_channels=3, num_classes=2).to(device)
student = StudentModel(in_channels=3, num_classes=2).to(device)

sample = torch.randn(16, 3, 32, 32).to(device)
with torch.no_grad():
    print("Teacher model stats:")
    pred = teacher(sample)
    for name, param in get_model_stats(teacher, sample.shape).items():
        print(f"{name}: {param}")
    print("Student model stats:")
    pred = student(sample)
    for name, param in get_model_stats(student, sample.shape).items():
        print(f"{name}: {param}")

Teacher model stats:


Unsupported operator aten::max_pool2d encountered 4 time(s)
Unsupported operator aten::max_pool2d encountered 3 time(s)


flops: 13267632128
params: 33995010
Student model stats:
flops: 1252130816
params: 949442


In [5]:
from models.unet import TeacherUNet
from models.unet import StudentUNet

teacher = TeacherUNet().to(device)
student = StudentUNet().to(device)

sample = torch.randn(16, 3, 32, 32).to(device)
with torch.no_grad():
    print("Teacher model stats:")
    pred = teacher(sample)
    for name, param in get_model_stats(teacher, sample.shape).items():
        print(f"{name}: {param}")
    print("Student model stats:")
    pred = student(sample)
    for name, param in get_model_stats(student, sample.shape).items():
        print(f"{name}: {param}")

Teacher model stats:


Unsupported operator aten::max_pool2d encountered 4 time(s)
Unsupported operator aten::max_pool2d encountered 3 time(s)


flops: 12058886144
params: 31043586
Student model stats:
flops: 1252130816
params: 949442


In [ ]:
def count_parameters(model):
    """Count trainable parameters in model"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def knowledge_distillation_loss(student_outputs, teacher_outputs, targets, alpha=0.5, temperature=2.0):
    """
    Combined loss function for knowledge distillation:
    - soft targets from teacher (KL divergence loss)
    - hard targets from ground truth (cross entropy loss)
    
    Args:
        student_outputs: logits from student model
        teacher_outputs: logits from teacher model
        targets: ground truth labels
        alpha: weight balancing hard vs soft targets (0.5 means equal weight)
        temperature: softening parameter for softmax
    """
    # Soft targets loss (KL divergence)
    soft_targets = F.softmax(teacher_outputs / temperature, dim=1)
    log_probs = F.log_softmax(student_outputs / temperature, dim=1)
    distillation_loss = F.kl_div(log_probs, soft_targets, reduction='batchmean') * temperature * temperature
    
    # Hard targets loss (cross entropy)
    ce_loss = F.cross_entropy(student_outputs, targets)
    
    # Combined loss
    return alpha * ce_loss + (1 - alpha) * distillation_loss


# Training function for distillation
def train_with_distillation(student_model, teacher_model, train_loader, optimizer, device, alpha=0.5, temperature=2.0):
    student_model.train()
    teacher_model.eval()  # Teacher is fixed during distillation
    
    total_loss = 0
    
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass with student
        student_output = student_model(data)
        
        # Get teacher predictions (no grad needed)
        with torch.no_grad():
            teacher_output = teacher_model(data)
        
        # Calculate distillation loss
        loss = knowledge_distillation_loss(
            student_output, 
            teacher_output, 
            target, 
            alpha=alpha, 
            temperature=temperature
        )
        
        # Backward and optimize
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)


if __name__ == '__main__':
    # Create models with different size ratios
    teacher = create_teacher_unet(in_channels=3, num_classes=2)
    student = create_student_unet(in_channels=3, num_classes=2, size_ratio=0.1)
    student_medium = create_student_unet(in_channels=3, num_classes=2, size_ratio=0.25)
    student_large = create_student_unet(in_channels=3, num_classes=2, size_ratio=0.5)
    
    # Print model sizes
    teacher_params = count_parameters(teacher)
    small_params = count_parameters(student)
    medium_params = count_parameters(student_medium)
    large_params = count_parameters(student_large)
    
    print(f"Teacher model parameters: {teacher_params:,}")
    print(f"Small student parameters: {small_params:,} (ratio: {teacher_params/small_params:.2f}x)")
    print(f"Medium student parameters: {medium_params:,} (ratio: {teacher_params/medium_params:.2f}x)")
    print(f"Large student parameters: {large_params:,} (ratio: {teacher_params/large_params:.2f}x)")
    
    # Sample usage of training loop
    """
    # Setup
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    teacher.to(device)
    student = student_medium.to(device)  # Choose which student to train
    
    # First train the teacher with standard cross-entropy loss
    # ...
    
    # Then train the student with distillation
    optimizer = torch.optim.Adam(student.parameters(), lr=0.001)
    
    for epoch in range(100):
        train_loss = train_with_distillation(
            student, 
            teacher, 
            train_loader, 
            optimizer, 
            device,
            alpha=0.5, 
            temperature=2.0
        )
        print(f"Epoch {epoch+1}, Loss: {train_loss:.4f}")
    """